In [ ]:
import os
from bigmodule import M

def initialize(context):
    from bigtrader.finance.commission import PerOrder
    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))

def handle_data(context, data):
    instrument = context.data["instruments"][0]

    # 获取过去五天的历史价格
    average_price = data.history(instrument, "close", 5, "1d").mean()
    # #获取当前价格
    current_price = data.current(instrument, "close")

    # 如果上一时间点价格高出五天平均价1%, 则全仓买入
    if current_price > average_price * 1.01:
        context.order_target_percent(instrument, 1.0)
    # 如果上一时间点价格低于五天平均价, 则空仓卖出
    elif current_price < average_price and context.get_position(instrument).current_qty > 0:
        context.order_target(instrument, 0)

m5 = M.bigtrader.v35(
    data={"instruments": ["600900.SH"]},
    # 模拟交易模式下，TRADING_DATE 为交易日期，回测模式下没有这个变量
    start_date=os.getenv("TRADING_DATE", "2024-01-01"),
    # 模拟交易模式下，TRADING_DATE 为交易日期，回测模式下没有这个变量
    end_date=os.getenv("TRADING_DATE", "2024-02-28"),
    # 初始化函数，只最开始运行一次，一般用于设置手续费等
    initialize=initialize,
    handle_data=handle_data,
    capital_base=1000000,
    # 买入指令在开盘时执行
    order_price_field_buy="open",
    # 卖出指令在收盘时执行
    order_price_field_sell="close",
)